# Station 2: Log trace — 15 points

Reconstruct a missing page incident from synthetic local logs. Identify what the evidence establishes and what remains unknown.

**Scope:** These are practice scenarios inferred from `urtuu.md`, not official competition tasks.

**Offline setup:** Python 3 kernel; all exercise code uses the standard library. Run cells from top to bottom. No downloads or internet are required.

**How to study:** Read the worked example, then attempt the three core exercises and optional extra practice before opening the solution section. Use a 30–40 minute timer. Record any recovered fragment exactly.

## Learn it visually



**How to read this:** Read left to right. GET is an action, `/rules` is the resource, and 200/403/404 describe outcomes. A log records these exchanges so you can reconstruct the sequence later.

**Try before continuing:** Does a logged DELETE with status 403 prove that the page was deleted?

<details><summary>Show the answer</summary>

No. The server refused the request. Look for a successful change and corroborating evidence.

</details>


## Mini-course: reading a web incident like a timeline

**What the guideline confirms:** A page is missing; use logs to trace the responsible actor. The log format and available tools are unspecified.

### Lesson 1 — Requests and responses
A browser is a client. It sends a request to a server, which responds. A request contains a method (action) and a path (resource). `GET /rules` asks to read the rules. POST often submits data; DELETE requests removal. Actual behavior depends on the application.

The response status summarizes the result: 200 means success, 204 means success with no response body, 403 means access refused, 404 means resource not found, and 500 means a server error. A request to delete something is not proof that deletion succeeded.

### Lesson 2 — Read each field
A timestamp says when an event was logged. An IP address identifies a network endpoint. A username identifies an account. A request ID links records belonging to one operation. A path identifies the requested resource. Keep these separate: an IP or account is not automatically a unique person.

### Lesson 3 — Reconstruct cause and effect
Start with the missing page, filter logs to that path, and sort by time. Find its last successful read and first failed read. Investigate changes between those events. Cross-check an audit log, which records actions such as deletion or permission changes. An unrelated 404 for an icon is usually weaker evidence than a successful deletion of the exact page.

See the expanded possibility bank below for recognition clues and first steps.

**Learn first:** methods, common statuses, filtering, sorting, and evidence-based explanations. **Extra practice:** query parameters (`?id=7`), URL encoding (`%20` is a space), session IDs, and time zones.


## More station possibilities

**Practice predictions, not official questions.** These are alternative ways the station description could be implemented, not a promise that every topic will appear. The study order below is a learning priority, not a probability ranking.

| Possible task | What might reveal it | First approach |
|---|---|---|
| Successful deletion | A page changes from readable to missing | Find the last successful read, first missing response, and a successful change to the exact resource. |
| Rename or move | The old path fails but another path appears | Look for rename/move audit events and compare resource IDs; a 404 alone does not prove deletion. |
| Permission change | The page exists but some users lose access | Compare account roles and access-control changes with successful reads from other accounts. |
| Deployment / routing mistake | Several paths fail after a deployment or restart | Correlate release, routing, and application logs; distinguish an account action from a broken release. |
| Suspicious login sequence | Repeated failures followed by a successful session | Link the session to later actions; failed attempts alone do not identify the actor who changed the page. |
| Shared IP, different sessions | Several accounts use one network address | Prefer authenticated account, session, and request identifiers over IP-only attribution. |
| Request-ID correlation | One operation appears in access, application, and audit logs | Join records by the shared ID and show the evidence chain from request to stored change. |
| Time zones / clock differences | Events seem to occur in an impossible order | Normalize explicit time-zone offsets; note any unknown clock skew instead of silently inventing a correction. |
| Rotated or unsorted logs | The relevant interval crosses files or midnight | Combine the provided files and sort full timestamps, including dates; keep original line references. |
| Encoded URL clue | Percent escapes or unusual query parameters | Preserve the raw URL and inspect a decoded copy. Confirm that the parameter belongs to the relevant request. |
| Misleading HTTP status | 202, redirects, or a 200 response containing an error | Check application outcomes: 202 means accepted for processing, not completed; a redirect changes the next location. |
| Repeated requests / retries | The same operation appears several times | Use operation IDs and outcomes to distinguish repeated log records, retries, and separate changes. |
| Missing dependency | The page HTML loads, but a script or API returns errors | Separate the document request from its dependent requests; identify what makes the page appear broken. |
| Noisy scans or unrelated 404s | Many errors involve icons or unrelated paths | Filter by the target resource and incident window before ranking suspicious activity. |
| Recover a fragment from evidence | An audit field, filename, or response includes a clue | Record its exact value and source line; do not treat a client-supplied claim as a verified server fact. |

**Study order:** Start with methods/statuses, the incident window, and request-ID correlation. Then practice alternate causes of a missing page. Time normalization and multi-file traces are useful stretch tasks.

**Watch out:** Report the narrowest supported conclusion: “account X performed operation Y” may be justified even when “person X did it” is not. Preserve evidence that contradicts your first theory.

## Worked example: status is evidence
A `DELETE /page` returning **403** was denied. A later `DELETE /page` returning **204** indicates a successful request under this mock service’s documented behavior. A subsequent **404** supports that the page is now missing. An IP identifies a network source, not necessarily a person.

For these exercises, assume synchronized clocks, complete logs, and a mock service where successful DELETE removes the page. Real systems require checking those assumptions.

In [ ]:
logs = [('10:00:01', '10.0.0.11', 'naraa', 'GET', '/home', 200), ('10:00:05', '10.0.0.12', 'bold', 'GET', '/rules', 200), ('10:01:00', '10.0.0.23', 'guest', 'DELETE', '/rules', 403), ('10:01:10', '10.0.0.44', 'admin', 'POST', '/login', 200), ('10:01:14', '10.0.0.44', 'admin', 'DELETE', '/rules', 204), ('10:01:15', '10.0.0.12', 'bold', 'GET', '/rules', 404), ('10:01:16', '10.0.0.44', 'admin', 'GET', '/rules', 404), ('10:02:00', '10.0.0.23', 'guest', 'GET', '/favicon.ico', 404)]
columns = ("time", "ip", "user", "method", "path", "status")
records = [dict(zip(columns, row)) for row in logs]
for r in records:
    print("{time} {ip:12} {user:6} {method:6} {path:14} {status}".format(**r))

## Your exercises

1. **Trace (8 minutes):** Which request most directly explains the missing `/rules` page? Give timestamp, IP, account, method, and status. Why is the guest’s request insufficient?
2. **Filter (8 minutes):** List only `/rules` events in chronological order. Count 404 responses by IP across all paths, then only for `/rules`. Is the IP with most 404s necessarily responsible?
3. **Cross-check (10 minutes):** Use the audit records below to find the request ID and recovered fragment for the successful deletion. Distinguish an account from the human operating it. What additional evidence would you request?

In [ ]:
audit = [
    {"time":"10:01:00", "ip":"10.0.0.23", "request_id":"r17", "action":"delete_denied", "fragment":None},
    {"time":"10:01:14", "ip":"10.0.0.44", "request_id":"r19", "action":"delete_page", "fragment":"TRACE"},
]
# Your analysis here

## Extra practice: recognize the variation

Allow 10–15 additional minutes. Explain your method before checking the answers at the end of this notebook.

1. **Moved or deleted?** At 10:00, audit request `r8` says `move resource=17 from=/rules to=/guide success=true`. At 10:01, `GET /rules` returns 404 and `GET /guide` returns 200. What is the best explanation?
2. **Time zones:** Event A is `2026-11-14T10:05:00+08:00`; event B is `2026-11-14T02:06:00Z`. Which is earlier, assuming synchronized clocks?
3. **Shared address:** Accounts `ana` and `bat` share IP `10.0.0.8`. A successful deletion audit record identifies session `s2`, which the login log associates with `bat`. What can and cannot you conclude?
4. **Accepted is not completed:** A DELETE returns 202, then its job log records `failed: permission denied`. A later GET returns 200. Did this deletion succeed?

---
# Solutions — stop here until you have attempted the exercises
Compare both your answer and your reasoning. Rerun the practice with different inputs after studying the solutions.

In [ ]:
from collections import Counter
related = sorted((r for r in records if r["path"] == "/rules"), key=lambda r:r["time"])
for r in related: print(r)
successful = [r for r in related if r["method"] == "DELETE" and 200 <= r["status"] < 300]
assert len(successful) == 1
incident = successful[0]
print("Likely causal request:", incident)
print("All 404s:", Counter(r["ip"] for r in records if r["status"] == 404))
print("Rules 404s:", Counter(r["ip"] for r in related if r["status"] == 404))
matching = [a for a in audit if a["time"] == incident["time"] and a["ip"] == incident["ip"]]
print("Audit match:", matching)

**Answers:** The successful DELETE is at 10:01:14 from 10.0.0.44 using account `admin`; its status is 204. The guest received 403. Request `r19` carries fragment `TRACE`. Counting errors alone does not establish causation. To identify a person, look for authenticated session records, device ownership, and possible shared or stolen credentials. A shared request ID would provide stronger correlation than time/IP alone.

## Extra practice answers

1. Resource 17 was moved to `/guide`; the old URL's 404 does not establish deletion. Cite the successful move and the later reads.
2. A is 02:05 UTC, so A precedes B by one minute. An unspecified clock skew would weaken this precise ordering.
3. The operation is associated with `bat`'s authenticated session `s2`. The shared IP does not select a person, and the records alone do not rule out shared or stolen credentials.
4. No successful deletion is shown. The request was accepted for processing, but the recorded job failed and the page remained readable.